# Vector-only contributions to hybrid search

Test one hypothesis: fixed vector retrieval contributes documents that do not match `maths`.
The mixed fixture contains **6,183** documents: **6,133 keyword-only**, **25 shared**, and **25 vector-only** for this query and `k=50`.
Vectors are engineered to demonstrate retrieval mechanics, not embedding quality.

The original fixture, index, evidence, and notebooks remain unchanged. This notebook targets only `hybrid-pagination-mixed-v2` on the existing service; it never redeploys a missing service.
Creating/uploading the new index requires separate approval. The full eight-case experiment makes roughly **6,400 primary reads**, plus inventories and boundary checks, and can take tens of minutes.

For another measurement run, **skip one-time preparation** and start from **Read-only reuse and fresh run**. Cases are independent: after a failure, deliberately run other cases or **Local contribution review**. Failed controls remain failed; this does not resolve the original keyword-pagination acceptance blocker.

## Prerequisites and explicit paths
Use the existing demo kernel after the original setup/deployment journey. Keep the existing isolated preview environment. No SDK/API pins, authentication, setup, or cleanup changes are required.

In [ ]:
import json
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import Markdown, display

demo = Path.cwd()
if demo.name == 'notebooks':
    demo = demo.parent
if not (demo / 'app' / 'fixture.py').is_file():
    raise FileNotFoundError('Run from this demo or its notebooks directory.')
sys.path.insert(0, str(demo))
python = demo / '.venv' / 'Scripts' / 'python.exe'
preview_python = demo / '.venv-preview' / 'Scripts' / 'python.exe'
env_file = demo / 'env' / '.env'
project = demo / 'app' / 'dotnet' / 'HybridPagination.csproj'
for required in (python, preview_python, env_file, project):
    if not required.is_file():
        raise FileNotFoundError(required)
fixture_dir = demo / 'outputs' / 'generated' / 'reproduction' / 'fixtures' / 'mixed-v2'
manifest_path = fixture_dir / 'manifest.json'

def run_command(command):
    result = subprocess.run(command, cwd=demo, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr, file=sys.stderr)
    result.check_returncode()

print('Mixed target: hybrid-pagination-mixed-v2; original target remains hybrid-pagination-debug.')

## Local generation and declared groups
This writes only the separate local fixture. Group labels below are declarations—not measured retrieval results.

In [ ]:
from app.fixture import generate, load_manifest

generate(fixture_dir, version=2)
manifest, documents = load_manifest(manifest_path)
print(json.dumps({key: len(ids) for key, ids in manifest['membership'].items()}, indent=2))
print('Indexed:', manifest['recordCount'], 'Keyword matches:', manifest['keywordMatchCount'])
print('Declared non-keyword examples:')
for document in documents[-3:]:
    print(document['id'], document['content'])

## Build the independent .NET reader
Build without restoring or changing pinned packages. If dependencies are missing, use the existing locked restore instructions before retrying.

In [ ]:
run_command(['dotnet', 'build', str(project), '--no-restore', '--nologo'])

## Separately approved one-time preparation
Only run after approval to create/upload the new index. Set the confirmation below explicitly. An absent/empty correctly defined mixed target is allowed; a populated target is refused before upload or schema update.
For an already prepared target, **skip this heading**. An all-successful upload still waiting for indexing can be finalized by the read-only heading; failed uploads cannot.

In [ ]:
CONFIRM_NEW_INDEX_PREPARATION = False
if not CONFIRM_NEW_INDEX_PREPARATION:
    raise ValueError('New-index preparation is not confirmed. Obtain approval, or skip this heading for reuse.')
from app.fixture import prepare
print(json.dumps(prepare(manifest_path, env_file), indent=2))

## Read-only reuse and fresh run
This checks the ready receipt, schema, exact count, and complete ID/content inventory. It makes read-only Azure requests; hidden vectors are not remotely hash-verified. Keep the index unchanged throughout measurement. Rerunning this heading selects a fresh evidence directory; old cases cannot be overwritten.

In [ ]:
from app.fixture import verify_prepared_fixture

print(json.dumps(verify_prepared_fixture(manifest_path, env_file), indent=2))
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
run_dir = demo / 'outputs' / 'generated' / 'reproduction' / 'mixed-runs' / run_id
run_dir.mkdir(parents=True, exist_ok=False)
common = ['--manifest', str(manifest_path), '--env-file', str(env_file), '--run-dir', str(run_dir)]
print('Fresh run:', run_dir)

## Python keyword control
Expected: exactly 6,158 keyword IDs, 1,232 populated pages, three final rows, no duplicates. A failure remains incomplete evidence.

In [ ]:
run_command([str(python), '-m', 'app.reproduce', '--case', 'keyword', *common])

## Independent .NET keyword control

In [ ]:
run_command(['dotnet', 'run', '--project', str(project), '--no-build', '--', '--case', 'keyword', *common])

## Python vector control
Expected: exactly 25 shared and 25 vector-only IDs. If the initial count is absent, scan the existing ten candidate pages and retain null count metrics.

In [ ]:
run_command([str(python), '-m', 'app.reproduce', '--case', 'vector', *common])

## Independent .NET vector control

In [ ]:
run_command(['dotnet', 'run', '--project', str(project), '--no-build', '--', '--case', 'vector', *common])

## Python stable hybrid
Observe the advertised count and all advertised pages, including empties. Do not prescribe the smaller union: the shared full-keyword matches are not guaranteed inside the internal 1,000-document keyword window.

In [ ]:
run_command([str(python), '-m', 'app.reproduce', '--case', 'hybrid', *common])

## Independent .NET stable hybrid

In [ ]:
run_command(['dotnet', 'run', '--project', str(project), '--no-build', '--', '--case', 'hybrid', *common])

## Smaller retrievable-count hybrid [PREVIEW]
Python Search 12.1.0b2 / API 2026-08-01-preview: recall 1,000 and `countRetrievableResults`. Requires this run's completed stable Python hybrid baseline. This changes SDK, API, and settings together—not an isolated SDK fix or production recommendation.

In [ ]:
run_command([str(preview_python), '-m', 'app.preview', 'run', '--scenario', 'retrievable', '--baseline', str(run_dir / 'python-hybrid.json'), *common])

## Expanded retrievable-count hybrid [PREVIEW]
Recall 6,158 is the full keyword count, not the indexed total. Acceptance target—not yet a measured result: 6,183 reported/unique IDs, 1,237 populated pages, three final rows, no duplicates/empty advertised pages/positive following-page probes.

In [ ]:
run_command([str(preview_python), '-m', 'app.preview', 'run', '--scenario', 'expanded', '--baseline', str(run_dir / 'python-hybrid.json'), *common])

## Local contribution review
This makes no replacement searches. Deliberately run it even after a failed case to retain an honest partial report. Missing or invalid controls leave verified additions unestablished and overall acceptance blocked.
Document text comes from the verified local corpus for actual hybrid-returned IDs, not the ID-only response. Exact internal keyword-window membership remains **not observed**, with null IDs. Counts, membership, and acceptance are separate findings.
JSON retains exact IDs, duplicate/missing IDs, boundary observations, case reasons, and source-file hashes. Review returns nonzero unless all required acceptance criteria pass.

In [ ]:
review = subprocess.run(
    [str(python), '-m', 'app.contributions', '--manifest', str(manifest_path), '--run-dir', str(run_dir)],
    cwd=demo, capture_output=True, text=True,
)
print(review.stdout)
print(review.stderr, file=sys.stderr)
report_path = run_dir / 'contribution-comparison.md'
display(Markdown(report_path.read_text(encoding='utf-8')))
print('Saved evidence:', run_dir)
review.check_returncode()